# SpecDist — Kaggle Quickstart

> **Full setup guide (GPU, secrets, models, checkpoint persistence, configs):** `docs/KAGGLE.md`

| Cell | What it does | When to run |
|------|-------------|-------------|
| 0. Bootstrap | Clone → deps → auth → launch pipeline | Once per fresh session |
| 1. Resume | Re-attach after restart — skips completed steps | After idle timeout / 9 h limit |
| 2b. Auto-backup | Snapshot checkpoints to a Kaggle Dataset every N min | Optional, alongside Cell 1 |
| 2. Monitor | State + log tail | Any time |
| 3. Save | Verify artifacts | Before session ends |

In [ ]:
# =============================================================================
# Cell 0 — BOOTSTRAP  (run once per fresh session)
# Edit the variables below, then run THIS CELL ONLY (do NOT use Run All /
# Shift+F5 — that also starts Cell 1 simultaneously → double pipeline → OOM).
# After a session restart use Cell 1 (Resume) instead — it is self-contained.
# Each run is TIERED: train + val_loss + light BE sanity (n=100, K=3, matched
# verifier). Heavy full-GSM8K eval is deferred to A100 confirmation.
# =============================================================================

# -- Edit these ---------------------------------------------------------------
REPO_URL     = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR     = "/kaggle/working/Distill-Spec-Research"
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = "/kaggle/working/specdist"

# Recommended accelerator: Settings -> Accelerator -> GPU P100 (single GPU).
# Do NOT use GPU T4 x2 -- it burns your SHARED 30 h/week Kaggle quota at 2x the
# rate (two GPUs) for work that fits in ONE 16 GB GPU. The teacher is 4-bit
# quantized (~4.5 GB) + student (~1.5 GB), so a single P100 is enough and gives
# ~2x the usable wall-clock time. Quota is 30 h/week, shared across P100 and
# T4 x2, resets Saturday 00:00 UTC. hw_scheduler.py auto-detects the GPU(s) and
# picks train/eval parallelism -- no config change needed when switching.

# CONFIG options:
#   kaggle     — Qwen3-8B (4-bit NF4, ~7.8 GB VRAM, 1000 steps, ~5-8 h)  ← best for Kaggle
#   colab      — Qwen3-4B (BF16,      ~10.7 GB VRAM, 500 steps,  ~4 h)    ← safe / no quant
#   colab_lite — Qwen3-1.7B (BF16,    ~5.6 GB VRAM,  300 steps,  ~25 min) ← quick trend check
CONFIG      = "kaggle"
SMOKE       = False    # True = 10-step crash check (~5 min); recommended on a new setup
BACKGROUND  = True     # True = pipeline runs in background; monitor with Cell 2
LOSSES      = None     # None = all losses  |  "kl,ebe" = subset
EXTRA_ARGS  = []

# Kaggle Models: attach Qwen3 directly from kaggle.com/models/qwen-lm/qwen-3
# In notebook editor: Add-ons → Add Model → search "qwen-3" → attach 0.6b and 8b variants
# Paths below are set automatically when the models are attached.
# Set to None to download from HuggingFace Hub instead (~5-10 min).
KAGGLE_DRAFT_MODEL  = "/kaggle/input/models/qwen-lm/qwen-3/transformers/0.6b/1"
KAGGLE_TARGET_MODEL = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"

# Optional: path to a pre-uploaded Kaggle dataset containing eval/training JSONL files.
# Upload once: python deploy/upload_to_kaggle.py --only data
# Attach to notebook → Add Data → Your Datasets → specdist-datasets
# Example: KAGGLE_DATA_DATASET = "/kaggle/input/specdist-datasets"
KAGGLE_DATA_DATASET = None

# GSM8K training data from Kaggle (thedevastator/grade-school-math-8k-q-a).
# Add Data → search "grade-school-math-8k" → attach → set path below.
# Converts main_train.csv to gsm8k_train.jsonl, skipping HuggingFace download.
KAGGLE_GSM8K_DATASET = "/kaggle/input/datasets/thedevastator/grade-school-math-8k-q-a"
# -----------------------------------------------------------------------------

import os, subprocess, sys

# --- GPU quota guard: warn if running on T4x2 (burns 2x the shared weekly quota) ---
try:
    import torch as _t
    if _t.cuda.is_available() and _t.cuda.device_count() >= 2:
        _names = {_t.cuda.get_device_properties(i).name for i in range(_t.cuda.device_count())}
        if any("T4" in n for n in _names):
            print("=" * 70)
            print("  WARNING: T4 x2 detected.")
            print("  This burns your SHARED 30 h/week Kaggle GPU quota at 2x the rate.")
            print("  Everything fits in ONE 16 GB GPU (teacher is 4-bit quantized).")
            print("  Recommended: Settings -> Accelerator -> GPU P100 (single GPU).")
            print("=" * 70)
except Exception:
    pass

def _prereq_token(name):
    """Read a platform secret before deploy_utils is available (needed for git clone)."""
    try:
        from kaggle_secrets import UserSecretsClient; v = UserSecretsClient().get_secret(name)
        if v: return v
    except Exception: pass
    return os.environ.get(name, "")

gh = _prereq_token("GITHUB_TOKEN")
if gh: os.environ["GITHUB_TOKEN"] = gh

clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0: print(r.stderr.strip()); raise subprocess.CalledProcessError(r.returncode, r.args)
    print("Repo cloned")
else:
    if gh: subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clone_url], capture_output=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main"], capture_output=True, text=True)
    r = subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], capture_output=True, text=True)
    print(r.stdout.strip() or "Synced to origin/main")
    if r.returncode != 0: print("[sync error]", r.stderr.strip())

sys.modules.pop("deploy_utils", None)
sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import bootstrap, run_pipeline

# Skip HF model download when Kaggle Models are attached (saves 5-10 min + 17 GB).
_models_on_kaggle = (
    os.path.isdir(KAGGLE_DRAFT_MODEL or "") and
    os.path.isdir(KAGGLE_TARGET_MODEL or "")
)
bootstrap(CONFIG, STORAGE_ROOT, REPO_DIR, gbv_dir=GBV_DIR,
          kaggle_data_dataset=KAGGLE_DATA_DATASET,
          kaggle_gsm8k_dataset=KAGGLE_GSM8K_DATASET,
          skip_model_prefetch=_models_on_kaggle)
_extra = list(EXTRA_ARGS)
if KAGGLE_DRAFT_MODEL and os.path.isdir(KAGGLE_DRAFT_MODEL):
    _extra += ["--draft",  KAGGLE_DRAFT_MODEL]
if KAGGLE_TARGET_MODEL and os.path.isdir(KAGGLE_TARGET_MODEL):
    _extra += ["--target", KAGGLE_TARGET_MODEL]
_ = run_pipeline(CONFIG, STORAGE_ROOT, GBV_DIR,
                 smoke=SMOKE, losses=LOSSES,
                 background=BACKGROUND, extra_args=_extra or None)


In [ ]:
# =============================================================================
# Cell 1 — RESUME  (after session restart, idle timeout, or 9-hour limit)
# Self-contained: works correctly even if Cell 0 never ran this session.
# The pipeline reads the state file and skips already-completed steps.
# =============================================================================

# -- Edit these (must match Cell 0) -------------------------------------------
REPO_URL            = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR            = "/kaggle/working/Distill-Spec-Research"
GBV_DIR             = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT        = "/kaggle/working/specdist"
CONFIG              = "kaggle"   # must match the CONFIG used in Cell 0
KAGGLE_DRAFT_MODEL  = "/kaggle/input/models/qwen-lm/qwen-3/transformers/0.6b/1"   # same as Cell 0
KAGGLE_TARGET_MODEL = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"     # same as Cell 0
KAGGLE_DATA_DATASET  = None       # same as Cell 0
KAGGLE_GSM8K_DATASET = "/kaggle/input/datasets/thedevastator/grade-school-math-8k-q-a"  # same as Cell 0
# -----------------------------------------------------------------------------

import os, subprocess, sys

def _prereq_token(name):
    """Read a platform secret before deploy_utils is available (needed for git clone)."""
    try:
        from kaggle_secrets import UserSecretsClient; v = UserSecretsClient().get_secret(name)
        if v: return v
    except Exception: pass
    return os.environ.get(name, "")

gh = _prereq_token("GITHUB_TOKEN")
if gh: os.environ["GITHUB_TOKEN"] = gh

clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0: print(r.stderr.strip()); raise subprocess.CalledProcessError(r.returncode, r.args)
    print("Repo cloned")
else:
    if gh: subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clone_url], capture_output=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main"], capture_output=True, text=True)
    r = subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], capture_output=True, text=True)
    print(r.stdout.strip() or "Synced to origin/main")
    if r.returncode != 0: print("[sync error]", r.stderr.strip())

sys.modules.pop("deploy_utils", None)
sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import bootstrap, run_pipeline

_models_on_kaggle = (
    os.path.isdir(KAGGLE_DRAFT_MODEL or "") and
    os.path.isdir(KAGGLE_TARGET_MODEL or "")
)
bootstrap(CONFIG, STORAGE_ROOT, REPO_DIR, gbv_dir=GBV_DIR,
          kaggle_data_dataset=KAGGLE_DATA_DATASET,
          kaggle_gsm8k_dataset=KAGGLE_GSM8K_DATASET,
          skip_model_prefetch=_models_on_kaggle,
          restore_checkpoints=True)
_extra = []
if KAGGLE_DRAFT_MODEL and os.path.isdir(KAGGLE_DRAFT_MODEL):
    _extra += ["--draft",  KAGGLE_DRAFT_MODEL]
if KAGGLE_TARGET_MODEL and os.path.isdir(KAGGLE_TARGET_MODEL):
    _extra += ["--target", KAGGLE_TARGET_MODEL]
print(f"Resuming {CONFIG} — completed steps are skipped automatically.\n")
_ = run_pipeline(CONFIG, STORAGE_ROOT, GBV_DIR, background=True,
                 extra_args=_extra or None)


In [ ]:
# =============================================================================
# Cell 2b — AUTO-BACKUP  (run AFTER Cell 1, in its own cell, leave running)
# Snapshots checkpoints -> Kaggle Dataset every INTERVAL_MIN minutes.
# Also doubles as a keep-alive. Interrupt the cell to stop (final backup runs).
# Secrets required: see docs/KAGGLE.md → "Persisting checkpoints"
# =============================================================================
import sys, subprocess

GBV_DIR      = "/kaggle/working/Distill-Spec-Research/gbv-research"
STORAGE_ROOT = "/kaggle/working/specdist"
DATASET_SLUG = None     # None -> "<KAGGLE_USERNAME>/specdist-checkpoints"
INTERVAL_MIN = 30       # back up every 30 min (LoRA adapters are small)

sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import backup_loop

# Find the running pipeline PID so the loop auto-stops (and does a final backup)
# when training+eval finish.
try:
    _pid = int(subprocess.run(["pgrep", "-f", "experiment.py"],
                              capture_output=True, text=True).stdout.split()[0])
except Exception:
    _pid = None

backup_loop(STORAGE_ROOT, DATASET_SLUG, interval_min=INTERVAL_MIN, pid=_pid)


In [ ]:
# =============================================================================
# Cell 2 — MONITOR  (safe to run any time, including while pipeline runs)
# Set AUTO_REFRESH = True for a live tail; interrupt the cell to stop.
# CONFIG must match the config used in Cell 0 or Cell 1.
# =============================================================================
import sys

REPO_DIR     = "/kaggle/working/Distill-Spec-Research"
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = "/kaggle/working/specdist"
CONFIG       = "kaggle"  # must match Cell 0/1
AUTO_REFRESH = False     # True = live tail loop (interrupt cell to stop)
REFRESH_SECS = 20

sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import monitor

monitor(STORAGE_ROOT, CONFIG, auto_refresh=AUTO_REFRESH, refresh_secs=REFRESH_SECS)

In [ ]:
# =============================================================================
# Cell 3 — SAVE  (verify artifacts before the session ends)
# Persistence options: see docs/KAGGLE.md → "Persisting checkpoints"
# =============================================================================
import os, json, pathlib

STORAGE_ROOT = "/kaggle/working/specdist"

ckpt_dir = os.path.join(STORAGE_ROOT, "checkpoints")
db_path  = os.path.join(STORAGE_ROOT, "results.db")
log_path = os.path.join(STORAGE_ROOT, "logs", "pipeline_output.log")

status = {
    "checkpoints": os.listdir(ckpt_dir) if os.path.isdir(ckpt_dir) else [],
    "results_db_bytes": os.path.getsize(db_path) if os.path.exists(db_path) else 0,
    "log_lines": sum(1 for _ in open(log_path)) if os.path.exists(log_path) else 0,
}
with open(os.path.join(STORAGE_ROOT, "run_status.json"), "w") as f:
    json.dump(status, f, indent=2)

print(f"Artifacts at: {STORAGE_ROOT}")
print(f"  results.db   : {status['results_db_bytes']:,} bytes")
print(f"  checkpoints/ : {status['checkpoints']}")
print(f"  log lines    : {status['log_lines']}")
print()

try:
    total = sum(
        f.stat().st_size
        for f in pathlib.Path("/kaggle/working").rglob("*")
        if f.is_file()
    )
    print(f"Total /kaggle/working/ usage: {total / 1024**3:.2f} GB (limit 20 GB)")
except Exception as e:
    print(f"Could not compute disk usage: {e}")

print()
print("Persistence options: see docs/KAGGLE.md -> 'Persisting checkpoints'")